# Работа с Excel

Материалы:
* Макрушин С.В. Лекция 7: Работа с Excel
* https://docs.xlwings.org/en/stable/quickstart.html
* https://nbviewer.jupyter.org/github/pybokeh/jupyter_notebooks/blob/master/xlwings/Excel_Formatting.ipynb#search_text


## Задачи для совместного разбора

1. На листе "Рецептура" файла `себестоимостьА_в1.xlsx` для области "Пшеничный хлеб" рассчитать себестоимость всех видов продукции.

In [160]:
import xlwings as xw
wb = xw.Book("себестоимостьА_в1.xlsx")
sht = wb.sheets("Рецептура")
prices  = sht.range("G14").expand("right")
costs = []
for i in range(7,11):
    cost = 0
    x = sht.range((i,7),(i,15))
    row = x.value
    for i in range(len(row)):
        if row[i] != None:
            cost += row[i] * prices.value[i] 
    costs.append(cost)
print(costs)

[21.48, 16.525, 17.423000000000002, 18.085]


2. Результаты расчетов 1.1 сохранить в отдельном столбце области "Пшеничный хлеб"

In [164]:
sht.range("P5").options(transpose=True).value = ["Себестоимость", ""] + costs

3. Приблизить форматирование столбца, добавленного в задаче 2 к оформлению всей области.

In [168]:
sht.range('P5').api.WrapText = True
sht.range('P5:P6').api.Font.Bold = True

4. Выполнить 3 с помощью "протягиваемых" формул.

## Лабораторная работа 7.1

1. Загрузите данные из файлов `reviews_sample.csv` (__ЛР2__) и `recipes_sample.csv` (__ЛР5__) в виде `pd.DataFrame`. Обратите внимание на корректное считывание столбца(ов) с индексами. Оставьте в таблице с рецептами следующие столбцы: `id`, `name`, `minutes`, `submitted`, `description`, `n_ingredients`

In [171]:
import pandas as pd
recipes = pd.read_csv('recipes_sample.csv', parse_dates=['submitted'])
recipies = recipes[["id", "name", "minutes", "submitted", "description", "n_ingredients"]]
reviews = pd.read_csv('reviews_sample.csv', index_col=0)
print(reviews)
recipies

            user_id  recipe_id        date  rating  \
370476        21752      57993  2003-05-01       5   
624300       431813     142201  2007-09-16       5   
187037       400708     252013  2008-01-10       4   
706134   2001852463     404716  2017-12-11       5   
312179        95810     129396  2008-03-14       5   
...             ...        ...         ...     ...   
1013457     1270706     335534  2009-05-17       4   
158736      2282344       8701  2012-06-03       0   
1059834      689540     222001  2008-04-08       5   
453285   2000242659     354979  2015-06-02       5   
691207       463435     415599  2010-09-30       5   

                                                    review  
370476   Last week whole sides of frozen salmon fillet ...  
624300   So simple and so tasty!  I used a yellow capsi...  
187037   Very nice breakfast HH, easy to make and yummy...  
706134   These are a favorite for the holidays and so e...  
312179   Excellent soup!  The tomato flavor is

,id,name,minutes,submitted,description,n_ingredients
0,44123,george s at the cove black bean soup,90,2002-10-25,an original recipe created by chef scott meska...,18.0
1,67664,healthy for them yogurt popsicles,10,2003-07-26,my children and their friends ask for my homem...,NaN
2,38798,i can t believe it s spinach,30,2002-08-29,"these were so go, it surprised even me.",8.0
3,35173,italian gut busters,45,2002-07-27,my sister-in-law made these for us at a family...,NaN
4,84797,love is in the air beef fondue sauces,25,2004-02-23,i think a fondue is a very romantic casual din...,NaN
...,...,...,...,...,...,...
29995,267661,zurie s holey rustic olive and cheddar bread,80,2007-11-25,this is based on a french recipe but i changed...,10.0
29996,386977,zwetschgenkuchen bavarian plum cake,240,2009-08-24,"this is a traditional fresh plum cake, thought...",11.0
29997,103312,zwiebelkuchen southwest german onion cake,75,2004-11-03,this is a traditional late summer early fall s...,NaN
29998,486161,zydeco soup,60,2012-08-29,this is a delicious soup that i originally fou...,NaN


2. Случайным образом выберите 5% строк из каждой таблицы и сохраните две таблицы на разные листы в один файл `recipes.xlsx`. Дайте листам названия "Рецепты" и "Отзывы", соответствующие содержанию таблиц. 

In [172]:
rand_rec = recipies.iloc[100:len(recipies)//100*5 + 100,:]
rand_rew = reviews.iloc[:len(reviews)//100*5,:]
with pd.ExcelWriter("recipes.xlsx") as writer:
    rand_rec.to_excel(writer, sheet_name = "Рецепты", engine='xlsxwriter')
    rand_rew.to_excel(writer, sheet_name = "Отзывы")


3. Используя `xlwings`, добавьте на лист `Рецепты` столбец `seconds_assign`, показывающий время выполнения рецепта в секундах. Выполните задание при помощи присваивания массива значений диапазону ячеек.

In [176]:
import xlwings as xw
wb = xw.Book("recipes.xlsx")
wb.sheets["Рецепты"].activate()
seconds = xw.Range("D2").expand("down")
xw.Range("H1").options(transpose=True). value = ["seconds_assign"]+[el*60 for el in seconds.value]

4. Используя `xlwings`, добавьте на лист `Рецепты` столбец `seconds_formula`, показывающий время выполнения рецепта в секундах. Выполните задание при помощи формул Excel.

In [202]:
import xlwings as xw
wb = xw.Book("recipes.xlsx")
wb.sheets["Рецепты"].activate()
fml = xw.Range("H2").formula = f'=D2 * 60'
xw.Range("H2:H1501").formula = fml

5. Сделайте названия всех добавленных столбцов полужирными и выровняйте по центру ячейки.

6. Раскрасьте ячейки столбца `minutes` в соответствии со следующим правилом: если рецепт выполняется быстрее 5 минут, то цвет - зеленый; от 5 до 10 минут - жёлтый; и больше 10 - красный.

In [ ]:
xw.sheets["Рецепты"].activate()
minutes = xw.Range("D2").expand("down")
for el in minutes:
    if el.value < 5:
        el.color = (0,255,0)
    elif el.value < 10:
        el.color = (255,255,0)
    else:
        el.color = (255,0,0)

7. Добавьте на лист `Рецепты`  столбец `n_reviews`, содержащий кол-во отзывов для этого рецепта. Выполните задание при помощи формул Excel.

In [223]:
xw.sheets["Рецепты"].activate()
ids_rec = xw.Range("B2").expand("down").value
xw.sheets["Отзывы"].activate()
ids_rew = xw.Range("C2").expand("down").value
count = []
for el in ids_rec:
    count.append(ids_rew.count(el))
xw.sheets["Рецепты"].activate()
xw.Range("I1").options(transpose = True).value = ['n_reviews'] + count

## Лабораторная работа 7.2

8. Напишите функцию `validate()`, которая проверяет соответствие всех строк из листа `Отзывы` следующим правилам:
    * Рейтинг - это число от 0 до 5 включительно
    * Соответствующий рецепт имеется на листе `Рецепты`
    
В случае несоответствия этим правилам, выделите строку красным цветом

In [185]:
import xlwings as xw
def validate():
    wb = xw.Book("recipes.xlsx")
    wb.sheets["Рецепты"].activate()
    rec_id = xw.Range("B2").expand("down").value
    xw.sheets["Отзывы"].activate()
    # rownum = xw.Range('A1').current_region.last_cell.row
    # for i in range(2, rownum + 1): 
    #     x = xw.Range((i, 1)).expand("right")
    #     rate = xw.Range((i, 5)).value
    #     id = xw.Range((i,3)).value
    #     if not (0 <= rate <= 5) or id not in rec_id:
    #         x.color = (255,0,0)
    table = xw.Range("A2").expand("table")
    for row in table.rows:
        rate = row.value[4]
        id = row.value[2]
        if not(0 <= rate <= 5) or id not in rec_id:
            row.color = (255,0,0)
validate()

9. В файле `recipes_model.csv` находится модель данных предметной области "рецепты". При помощи пакета `csv` считайте эти данные. При помощи пакета `xlwings` запишите данные на лист `Модель` книги `recipes_model.xlsx`, начиная с ячейки `A2`, не используя циклы. Сделайте скриншот текущего состояния листа и прикрепите в ячейку ноутбука. 

10. При помощи пакета `xlwings` добавьте в столбец J формулу для описания столбца на языке SQL. Формула должна реализовывать следующую логику:

    1\. в начале строки идут значения из столбцов В и C (значение столбца С приведено к верхнему регистру), разделенные пробелом
    
    2\. далее идут слова на основе столбца "Ключ"
        2.1 если в столбце "Ключ" указано значение "PK", то дальше через пробел идет ключевое слово "PRIMARY KEY"
        2.2 если в столбце "Ключ" указано значение "FK", то дальше через пробел идет ключевое слово "REFERENCES", затем значения столбцов H и I в формате "название_таблицы(название_столбца)"
        
    3\. если в столбце "Обязательно к заполнению" указано значение "Y" и в столбце "Ключ" указано не "PK", то дальше через пробел идет ключевое слово "NOT NULL".

Заполните этой формулой необходимое количество строк, используя "протягивание". Количество строк для протягивания определите на основе данных.

Сделайте скриншот текущего состояния листа и прикрепите в ячейку ноутбука.

11. При помощи пакета `xlwings` измените стилизацию листа `Модель`.
* для заголовков добавьте заливку цвета `00ccff`
* примените автоподбор ширины столбца;
* сделайте шрифт заголовков полужирным;
* добавьте таблице автофильтр.

Сделайте скриншот текущего состояния листа и прикрепите в ячейку ноутбука.

12. Посчитайте количество атрибутов для каждой из сущностей. Создайте лист `Статистика` и запишите в него результат группировки, начиная с ячейки "А1". Визуализируйте полученный результат при помощи столбчатой диаграммы. Сохраните полученную визуализацию на лист `Статистика`, начиная с ячейки "E2".  Сделайте скриншот листа `Статистика` и прикрепите в ячейку ноутбука.

* Вы можете воспользоваться методами для визуализации, которые поставляются вместе с объектами `pandas` (см. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot) 